# Tumblr Tag Scraper
### Based on [bernorieder/TumblrTool](https://github.com/bernorieder/TumblrTool)

Collects Tumblr posts for a given **search tag** via the Tumblr v2 `/tagged` API and saves them to a **CSV file**.

Supports optional **date range filtering** via `DATE_FROM` / `DATE_TO`.

---
Get a **free Tumblr API key** (Consumer Key) at https://www.tumblr.com/oauth/apps  
No login/OAuth required for read-only tag searches.


In [1]:
%pip install requests -q
print("requests ready ✓")


Note: you may need to restart the kernel to use updated packages.
requests ready ✓


In [1]:
import csv
import logging
import os
import re
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import requests

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("tumblr_scraper")
print("Imports OK ✓")


Imports OK ✓


## 1 · API Key

Fill in your Tumblr Consumer Key below, **or** set `TUMBLR_API_KEY` as an environment variable.

Get a key: https://www.tumblr.com/oauth/apps → *Register application*


In [ ]:
API_KEY = ""   # e.g. "aBcDeFgHiJkLmNoPqRsTuVwXyZ0123456789aBcD"
API_KEY = API_KEY or os.environ.get("TUMBLR_API_KEY")
assert API_KEY, "No API key found. Set API_KEY above or export TUMBLR_API_KEY=your_key"
print("API key loaded ✓")


API key loaded ✓


## 2 · Configuration

Set your tag, date range, and output options below.

**Date range** (`DATE_FROM` / `DATE_TO`):
- Set both to scrape a specific window, e.g. a single year or month.
- Set only `DATE_TO` to scrape everything up to that date.
- Set only `DATE_FROM` to scrape everything from that date forward.
- Leave both as `None` to scrape the most recent posts up to `LIMIT`.

Dates are **inclusive** and interpreted as **UTC midnight**.


In [3]:
# ── Search parameters ─────────────────────────────────────────────────────────
TAG       = "ao3"           # Tag to search — no # needed; spaces are fine
LIMIT     = 10_000          # Safety cap on total posts (useful for open-ended runs)

# ── Date range (YYYY-MM-DD strings, or None to skip) ──────────────────────────
DATE_FROM = "2026-05-09"            # e.g. "2023-01-01"   — inclusive lower bound
DATE_TO   = "2026-05-13"            # e.g. "2023-12-31"   — inclusive upper bound

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path(".")

# ── API behaviour ─────────────────────────────────────────────────────────────
POSTS_PER_REQUEST = 20
REQUEST_DELAY     = 0.5

# ── Derived output filename ────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
safe_tag = re.sub(r"[^\w\-]", "_", TAG.lower()).strip("_")
run_ts   = datetime.now(tz=timezone.utc).strftime("%Y%m%d_%H%M%S")

if DATE_FROM or DATE_TO:
    from_str = DATE_FROM.replace("-","") if DATE_FROM else "start"
    to_str   = DATE_TO.replace("-","")   if DATE_TO   else "now"
    OUTPUT_FILE = OUTPUT_DIR / f"tumblr_{safe_tag}_{from_str}_to_{to_str}_{run_ts}.csv"
else:
    OUTPUT_FILE = OUTPUT_DIR / f"tumblr_{safe_tag}_{LIMIT}posts_{run_ts}.csv"

print(f"Tag       : {TAG}")
print(f"Date from : {DATE_FROM or '(none)'}")
print(f"Date to   : {DATE_TO   or '(none)'}")
print(f"Limit     : {LIMIT}")
print(f"Output    : {OUTPUT_FILE}")


Tag       : ao3
Date from : 2026-05-09
Date to   : 2026-05-13
Limit     : 10000
Output    : tumblr_ao3_20260509_to_20260513_20260514_211544.csv


## 3 · Helper Functions


In [4]:
CSV_COLUMNS = [
    "post_id", "post_type", "blog_name", "date", "timestamp", "datetime_utc",
    "post_url", "note_count", "tags", "title", "body_or_caption",
    "source_url", "source_title", "media_url", "short_url",
]

# ── Date utilities ────────────────────────────────────────────────────────────
def parse_date(date_str, label):
    """Parse YYYY-MM-DD → UTC-aware datetime. Raises ValueError on bad input."""
    try:
        dt = datetime.strptime(date_str.strip(), "%Y-%m-%d")
        return dt.replace(tzinfo=timezone.utc)
    except ValueError:
        raise ValueError(f"{label} '{date_str}' is not valid. Use YYYY-MM-DD.")

def dt_to_ts(dt):
    return int(dt.timestamp())

def ts_to_dt(ts):
    return datetime.fromtimestamp(float(ts), tz=timezone.utc)

def to_iso(ts):
    return datetime.fromtimestamp(float(ts), tz=timezone.utc).isoformat()

# ── Text utilities ────────────────────────────────────────────────────────────
def clean(text):
    if not text:
        return ""
    return re.sub(r"[\r\n\t]+", " ", str(text)).strip()

def extract_media_url(post):
    ptype = post.get("type", "")
    if ptype == "photo":
        photos = post.get("photos", [])
        if photos:
            return photos[0].get("original_size", {}).get("url", "")
    elif ptype == "video":
        return post.get("thumbnail_url", "")
    elif ptype == "audio":
        return post.get("album_art", "")
    elif ptype == "link":
        return post.get("link_image", "")
    return ""

def extract_text_content(post):
    ptype = post.get("type", "")
    if ptype == "text":
        return clean(post.get("body", ""))
    elif ptype == "quote":
        return clean(post.get("text", ""))
    elif ptype == "link":
        return clean(post.get("description", ""))
    elif ptype == "chat":
        dialogue = post.get("dialogue", [])
        return clean(" | ".join(f"{d.get('label','')} {d.get('phrase','')}" for d in dialogue))
    else:
        return clean(post.get("caption", ""))

def flatten_post(post):
    tags = post.get("tags", [])
    return {
        "post_id":         post.get("id", ""),
        "post_type":       post.get("type", ""),
        "blog_name":       clean(post.get("blog_name", "")),
        "date":            post.get("date", ""),
        "timestamp":       post.get("timestamp", ""),
        "datetime_utc":    to_iso(post["timestamp"]) if post.get("timestamp") else "",
        "post_url":        post.get("post_url", ""),
        "note_count":      post.get("note_count", 0),
        "tags":            " | ".join(str(t) for t in tags),
        "title":           clean(post.get("title", "")),
        "body_or_caption": extract_text_content(post),
        "source_url":      post.get("source_url", ""),
        "source_title":    clean(post.get("source_title", "")),
        "media_url":       extract_media_url(post),
        "short_url":       post.get("short_url", ""),
    }

print("Helper functions defined ✓")


Helper functions defined ✓


## 4 · API Fetch Function


In [5]:
TUMBLR_TAGGED_URL = "https://api.tumblr.com/v2/tagged"

def fetch_page(tag, api_key, before=None):
    """Fetch one page (≤20 posts) from /tagged. Returns list of raw post dicts."""
    params = {"tag": tag, "limit": POSTS_PER_REQUEST, "api_key": api_key}
    if before is not None:
        params["before"] = before
    try:
        resp = requests.get(TUMBLR_TAGGED_URL, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()
    except requests.RequestException as exc:
        log.warning(f"Request error: {exc} — retrying after 2 s")
        time.sleep(2)
        return []
    if data.get("meta", {}).get("status") != 200:
        log.error(f"API error: {data.get('meta',{}).get('msg','unknown')}")
        return []
    return data.get("response", [])

print("fetch_page() defined ✓")


fetch_page() defined ✓


## 5 · Scrape

Paginates backwards through the tag feed using timestamps.

When `DATE_TO` is set, the initial cursor is placed just after that date so
the first API request already starts at the right point in the timeline.
When `DATE_FROM` is set, the loop stops as soon as a post older than that date
is encountered — making date-bounded runs significantly faster.


In [6]:
# ── Parse dates ───────────────────────────────────────────────────────────────
date_from = parse_date(DATE_FROM, "DATE_FROM") if DATE_FROM else None
date_to   = parse_date(DATE_TO,   "DATE_TO")   if DATE_TO   else None

if date_from and date_to and date_from > date_to:
    raise ValueError(f"DATE_FROM ({DATE_FROM}) must be before DATE_TO ({DATE_TO})")

ts_from = dt_to_ts(date_from) if date_from else None
ts_to   = dt_to_ts(date_to)   if date_to   else None

# Seed cursor just past DATE_TO so the first page is already in range
initial_before = (ts_to + 1) if ts_to else None

# ── Log header ─────────────────────────────────────────────────────────────────
clean_tag = TAG.lstrip("#")
if date_from or date_to:
    log.info(f'Scraping "{clean_tag}" | {DATE_FROM or "start"} → {DATE_TO or "now"} | cap={LIMIT}')
else:
    log.info(f'Scraping up to {LIMIT} posts tagged "{clean_tag}" …')

# ── Main loop ─────────────────────────────────────────────────────────────────
rows     = []
seen_ids = set()
before   = initial_before
page     = 0

while len(rows) < LIMIT:
    page += 1
    cursor_label = ts_to_dt(before).strftime("%Y-%m-%d %H:%M") if before else "now"
    log.info(f"  Page {page:3d} | cursor={cursor_label} | collected: {len(rows)}")

    posts = fetch_page(clean_tag, API_KEY, before=before)

    if not posts:
        log.info("Empty response — no more posts available.")
        break

    new_count   = 0
    passed_from = False

    for post in posts:
        if len(rows) >= LIMIT:
            break

        pid = post.get("id")
        if pid in seen_ids:
            continue
        seen_ids.add(pid)

        post_ts = post.get("timestamp", 0)

        # Upper bound check
        if ts_to and post_ts > ts_to:
            continue

        # Lower bound check — stop as soon as we go past date_from
        if ts_from and post_ts < ts_from:
            passed_from = True
            log.info(f"  Reached posts older than {DATE_FROM} — stopping.")
            break

        rows.append(flatten_post(post))
        new_count += 1

    log.info(f"         → {new_count} new posts added.")

    if passed_from:
        break

    oldest_ts = posts[-1].get("timestamp")
    if oldest_ts is None or oldest_ts == before:
        log.info("Cursor did not advance — stopping.")
        break
    before = oldest_ts

    if len(posts) < POSTS_PER_REQUEST:
        log.info("Partial page — likely reached end of tag feed.")
        break

    time.sleep(REQUEST_DELAY)

print(f"\nScrape complete. Posts collected: {len(rows)}")


17:16:07  Scraping "ao3" | 2026-05-09 → 2026-05-13 | cap=10000
17:16:07    Page   1 | cursor=2026-05-13 00:00 | collected: 0
17:16:07           → 20 new posts added.
17:16:08    Page   2 | cursor=2026-05-12 23:01 | collected: 20
17:16:08           → 20 new posts added.
17:16:09    Page   3 | cursor=2026-05-12 22:06 | collected: 40
17:16:09           → 20 new posts added.
17:16:09    Page   4 | cursor=2026-05-12 21:11 | collected: 60
17:16:10           → 20 new posts added.
17:16:10    Page   5 | cursor=2026-05-12 20:03 | collected: 80
17:16:11           → 20 new posts added.
17:16:11    Page   6 | cursor=2026-05-12 19:12 | collected: 100
17:16:11           → 20 new posts added.
17:16:12    Page   7 | cursor=2026-05-12 18:19 | collected: 120
17:16:12           → 20 new posts added.
17:16:13    Page   8 | cursor=2026-05-12 17:06 | collected: 140
17:16:13           → 20 new posts added.
17:16:13    Page   9 | cursor=2026-05-12 16:03 | collected: 160
17:16:14           → 20 new posts added


Scrape complete. Posts collected: 245


## 6 · Save to CSV


In [7]:
with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(rows)

size_kb = OUTPUT_FILE.stat().st_size / 1024
print(f"✓  Saved {len(rows)} posts → {OUTPUT_FILE}  ({size_kb:.1f} KB)")


✓  Saved 245 posts → tumblr_ao3_20260509_to_20260513_20260514_211544.csv  (365.2 KB)


## 7 · Quick Inspection


In [ ]:
from collections import Counter

post_types  = Counter(r["post_type"] for r in rows)
note_counts = [int(r["note_count"]) for r in rows if r["note_count"] != ""]
all_tags    = [t.strip() for r in rows for t in r["tags"].split("|") if t.strip()]
top_tags    = Counter(t.lower() for t in all_tags).most_common(10)

print(f"Total posts      : {len(rows)}")
print(f"Post type counts : {dict(post_types)}")
if note_counts:
    print(f"Avg note count   : {sum(note_counts)/len(note_counts):.0f}")
    print(f"Max note count   : {max(note_counts):,}")
if rows:
    earliest = min(rows, key=lambda r: r["timestamp"] or 9e18)
    latest   = max(rows, key=lambda r: r["timestamp"] or 0)
    print(f"Earliest post    : {earliest['datetime_utc'][:10]}")
    print(f"Latest post      : {latest['datetime_utc'][:10]}")
print(f"\nTop 10 co-occurring tags:")
for tag_name, count in top_tags:
    print(f"  {count:5d}  {tag_name}")


In [ ]:
print(f"{'─'*70}")
for i, row in enumerate(rows[:3], 1):
    print(f"\n--- Post {i} ---")
    for col in CSV_COLUMNS:
        val = str(row.get(col, ""))
        print(f"  {col:<18} {val[:100]}")


## 8 · Reload from CSV (optional)


In [ ]:
import csv as _csv

# Adjust the path to your saved file
# load_path = Path("tumblr_ao3_20230101_to_20231231_20240101_120000.csv")
# with open(load_path, encoding="utf-8", newline="") as f:
#     rows = list(_csv.DictReader(f))
# print(f"Loaded {len(rows)} posts from {load_path}")
